# Simulated data — deep inspection

Detailed audit of a single simulation run. Every plot is interactive
(matplotlib via `ipympl`): drag to pan, scroll/zoom box to zoom, the home
button on the toolbar resets.

Sections:

1. Setup + load
2. Sanity checks
3. Store-level financial composition (cash, inventory value, outstanding-order value, equity, cumulative P&L)
4. Per-store decision signals (active SKUs, promotions, orders, activations / deactivations, median pricing)
5. Promotion + activation timelines (heatmaps)
6. Pricing dynamics
7. Inventory / sales / demand flow
8. Per-(store, product) drill-down
9. Product- and category-level profitability
10. Fulfilment quality (demand vs sales)
11. Final snapshot

If a plot does not render, run `pip install ipympl` (we add it via `uv add ipympl`)
and restart the kernel.


In [ ]:
# Interactive matplotlib backend — zoom + pan via the toolbar.
%matplotlib widget
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path
import sys; sys.path.append('../')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib import cm
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 120


## 1. Load

In [ ]:
# Pick the most recently modified simulation output under ../data/.
repo_root = Path.cwd().resolve().parent
data_root = repo_root / 'data'

sim_dirs = [p for p in data_root.iterdir() if p.is_dir() and (p / 'data').is_dir()]
if not sim_dirs:
    raise FileNotFoundError('No simulation output folders found under ../data')

latest_sim_dir = max(sim_dirs, key=lambda p: p.stat().st_mtime)
data_dir = latest_sim_dir / 'data'
print(f'Using simulation folder: {latest_sim_dir}')
print('Files:', sorted(p.name for p in data_dir.glob('*')))


In [ ]:
# Static catalog + store tables.
products_df = pd.read_parquet(data_dir / 'products.parquet')
stores_df = pd.read_parquet(data_dir / 'stores.parquet')

# Per-step / per-store / per-product time-series (long form).
ts = pd.read_parquet(data_dir / 'timeseries.parquet')

# Join unit_cost / base_price so we can value inventory + outstanding orders
# at cost and compare realised price against MSRP.
ts = ts.merge(
    products_df[['product_id', 'name', 'category', 'base_price', 'unit_cost']],
    on='product_id', how='left',
)

# Run-log holds time-series state that's not in the parquet: cash balance,
# active-product count per step, capacity, market supply/demand.
with open(data_dir / 'run_log.json') as f:
    run_log = json.load(f)

sim_steps = run_log['global']['time']['simulation_step']
sim_dates = pd.to_datetime(run_log['global']['time']['simulation_date'])

# Run-log keys stores as strings; parquet store_id may be int — keep the
# parquet dtype for store_ids and stringify when indexing run_log.
store_ids = sorted(ts['store_id'].unique().tolist())
print(f'Stores: {store_ids}')
print(f'Steps: {len(sim_steps)}  ({sim_dates.min().date()} → {sim_dates.max().date()})')
print(f'Products: {ts["product_id"].nunique()}')


## 2. Sanity checks

In [ ]:
summary_df = pd.DataFrame([
    {'table': 'stores', 'rows': len(stores_df), 'cols': stores_df.shape[1],
     'null_values': int(stores_df.isna().sum().sum())},
    {'table': 'products', 'rows': len(products_df), 'cols': products_df.shape[1],
     'null_values': int(products_df.isna().sum().sum())},
    {'table': 'timeseries', 'rows': len(ts), 'cols': ts.shape[1],
     'null_values': int(ts.isna().sum().sum())},
])
summary_df


In [ ]:
checks = {
    'negative_inventory_rows': int((ts['inventory'] < 0).sum()),
    'negative_sales_rows': int((ts['sales'] < 0).sum()),
    'negative_price_rows': int((ts['price'] < 0).sum()),
    # sales > inventory is OK because the row's `inventory` is the
    # *post-sale* on-hand value — track anyway as smoke signal.
    'sales_gt_inventory_rows': int((ts['sales'] > ts['inventory']).sum()),
    'price_below_unit_cost_rows': int((ts['price'] < ts['unit_cost']).sum()),
    'duplicate_(store,product,step)_rows': int(ts.duplicated(['store_id', 'product_id', 'simulation_step']).sum()),
    'unique_promotion_statuses': sorted(ts['promotion_status'].unique().tolist()),
    'unique_active_statuses': sorted(ts['active_status'].unique().tolist()),
}
pd.Series(checks, name='value')


## 3. Store financial composition

Total equity at each step is

```
equity = cash + inventory_at_cost + outstanding_orders_at_cost
```

The store pays for an order the moment it's placed (cash drops by
`order_qty × unit_cost`) but the goods aren't yet on hand — the
``outstanding_orders × unit_cost`` term keeps the equity curve
continuous across the delivery lag.

In [ ]:
# Cash balance per (store, step) from the run log. run_log keys are str;
# stringify the parquet store_id when indexing.
cash_rows = []
for sid in store_ids:
    bal = run_log['stores'][str(sid)]['balance']
    for t, step in enumerate(sim_steps):
        cash_rows.append({'store_id': sid, 'simulation_step': step, 'cash': bal[t]})
cash_df = pd.DataFrame(cash_rows)

val = ts.assign(
    inventory_value=ts['inventory'] * ts['unit_cost'],
    outstanding_value=ts['outstanding_orders'] * ts['unit_cost'],
)
store_step = (
    val.groupby(['store_id', 'simulation_step'], as_index=False)
       .agg(inventory_value=('inventory_value', 'sum'),
            outstanding_value=('outstanding_value', 'sum'),
            step_profit=('profit', 'sum'),
            revenue=('revenue', 'sum'),
            total_cost=('total_cost', 'sum'),
            holding_cost=('holding_cost', 'sum'))
)
store_step = store_step.merge(cash_df, on=['store_id', 'simulation_step'])
store_step['equity'] = store_step['cash'] + store_step['inventory_value'] + store_step['outstanding_value']
store_step = store_step.sort_values(['store_id', 'simulation_step'])
store_step['cumulative_profit'] = store_step.groupby('store_id')['step_profit'].cumsum()
store_step.head()


In [ ]:
# One figure per store: stacked equity components + cumulative P&L overlay
# on a twin axis. Drawn separately so each can be zoomed independently.
for sid in store_ids:
    g = store_step[store_step['store_id'] == sid]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.stackplot(
        g['simulation_step'],
        g['cash'], g['inventory_value'], g['outstanding_value'],
        labels=['Cash', 'Inventory value (cost)', 'Outstanding orders (cost)'],
        colors=['#1f77b4', '#2ca02c', '#ff7f0e'], alpha=0.85,
    )
    ax.plot(g['simulation_step'], g['equity'], color='black', lw=2.2, ls='--',
            label='Total equity')
    ax.set_xlabel('Simulation step')
    ax.set_ylabel('Equity components')
    ax.set_title(f'Store {sid} — equity composition + cumulative P&L')

    ax2 = ax.twinx()
    ax2.plot(g['simulation_step'], g['cumulative_profit'], color='crimson', lw=2.2,
             label='Cumulative P&L')
    ax2.set_ylabel('Cumulative P&L', color='crimson')
    ax2.tick_params(axis='y', labelcolor='crimson')
    ax2.grid(False)

    # Combine legends from both axes.
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)
    fig.tight_layout()
    plt.show()

In [ ]:
# Per-step P&L decomposition: revenue (positive bar) vs total_cost
# (negative bar) vs step profit overlay. Holding cost is part of total
# cost, plotted on a stacked-negative slot for visibility.
for sid in store_ids:
    g = store_step[store_step['store_id'] == sid]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.bar(g['simulation_step'], g['revenue'], color='#2ca02c', alpha=0.9,
           label='Revenue')
    # Order cost = total_cost - holding_cost; stack so both layers are visible.
    order_cost = g['total_cost'] - g['holding_cost']
    ax.bar(g['simulation_step'], -order_cost, color='#d62728', alpha=0.9,
           label='Order cost')
    ax.bar(g['simulation_step'], -g['holding_cost'], bottom=-order_cost,
           color='#9467bd', alpha=0.9, label='Holding cost')
    ax.plot(g['simulation_step'], g['step_profit'], color='black', lw=2,
            marker='o', ms=3, label='Step P&L')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel('Simulation step')
    ax.set_ylabel('Per-step amount')
    ax.set_title(f'Store {sid} — revenue vs costs per step')
    ax.legend(loc='upper left', fontsize=9)
    fig.tight_layout()
    plt.show()

## 4. Per-store decision signals

Each store gets its own one-row-per-signal figure: active SKU count,
activation/deactivation events, SKUs on promo, promo start/end events,
total order quantity, outstanding orders, median active-SKU price/MSRP
ratio.

In [ ]:
# Detect transitions per (store, product) for activate/deactivate/promo flows.
ts_sorted = ts.sort_values(['store_id', 'product_id', 'simulation_step']).copy()
ts_sorted['prev_active'] = ts_sorted.groupby(['store_id', 'product_id'])['active_status'].shift()
ts_sorted['prev_promo'] = ts_sorted.groupby(['store_id', 'product_id'])['promotion_status'].shift()

is_promo = (ts_sorted['promotion_status'] != 'Regular Price')
was_promo = (ts_sorted['prev_promo'] != 'Regular Price') & ts_sorted['prev_promo'].notna()

ts_sorted['activated'] = (ts_sorted['active_status'] == True) & (ts_sorted['prev_active'] == False)
ts_sorted['deactivated'] = (ts_sorted['active_status'] == False) & (ts_sorted['prev_active'] == True)
ts_sorted['promo_started'] = is_promo & ~was_promo
ts_sorted['promo_ended'] = ~is_promo & was_promo
ts_sorted['on_promo'] = is_promo

decisions_per_store = (
    ts_sorted.groupby(['store_id', 'simulation_step'], as_index=False)
             .agg(active_count=('active_status', lambda s: int(s.sum())),
                  activated=('activated', 'sum'),
                  deactivated=('deactivated', 'sum'),
                  on_promo=('on_promo', 'sum'),
                  promo_started=('promo_started', 'sum'),
                  promo_ended=('promo_ended', 'sum'),
                  order_qty=('order_quantity', 'sum'),
                  outstanding=('outstanding_orders', 'sum'))
)
active_only = ts_sorted[ts_sorted['active_status'] == True].copy()
active_only['price_ratio'] = active_only['price'] / active_only['base_price']
pr = (active_only.groupby(['store_id', 'simulation_step'], as_index=False)
      .agg(median_price_ratio=('price_ratio', 'median'),
           mean_price_ratio=('price_ratio', 'mean')))
decisions_per_store = decisions_per_store.merge(pr, on=['store_id', 'simulation_step'], how='left')
decisions_per_store.head()


In [ ]:
# One figure per signal, one subplot row per store, sharing the x-axis.
def _per_store_panel(value_fn, title, ylabel, kind='line', extra=None):
    fig, axes = plt.subplots(len(store_ids), 1, figsize=(11, 2.5 * len(store_ids)),
                              sharex=True, squeeze=False)
    for i, sid in enumerate(store_ids):
        ax = axes[i, 0]
        g = decisions_per_store[decisions_per_store['store_id'] == sid]
        value_fn(ax, g)
        ax.set_ylabel(f'Store {sid}\n{ylabel}', fontsize=9)
        ax.grid(True, alpha=0.3)
        if extra is not None:
            extra(ax, g)
    axes[-1, 0].set_xlabel('Simulation step')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

_per_store_panel(
    lambda ax, g: ax.plot(g['simulation_step'], g['active_count'], color='#1f77b4', lw=1.5),
    'Active SKU count per store', 'active SKUs',
)


In [ ]:
# Activations (positive) and deactivations (negative) per step.
def _act_deact(ax, g):
    ax.bar(g['simulation_step'], g['activated'], color='#2ca02c',
           label='Activated' if not ax.get_legend_handles_labels()[1] else None)
    ax.bar(g['simulation_step'], -g['deactivated'], color='#d62728',
           label='Deactivated' if 'Deactivated' not in ax.get_legend_handles_labels()[1] else None)
    ax.axhline(0, color='gray', lw=0.5)

fig, axes = plt.subplots(len(store_ids), 1, figsize=(11, 2.5 * len(store_ids)),
                          sharex=True, squeeze=False)
for i, sid in enumerate(store_ids):
    ax = axes[i, 0]
    g = decisions_per_store[decisions_per_store['store_id'] == sid]
    ax.bar(g['simulation_step'], g['activated'], color='#2ca02c', label='Activated')
    ax.bar(g['simulation_step'], -g['deactivated'], color='#d62728', label='Deactivated')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_ylabel(f'Store {sid}\nevents', fontsize=9)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(loc='upper right', fontsize=9)
axes[-1, 0].set_xlabel('Simulation step')
fig.suptitle('Activations / deactivations per step, per store')
fig.tight_layout()
plt.show()


In [ ]:
# SKUs on promotion per step.
_per_store_panel(
    lambda ax, g: ax.plot(g['simulation_step'], g['on_promo'], color='#9467bd', lw=1.5),
    'SKUs currently on promotion per store', 'on promo',
)


In [ ]:
# Promo starts (positive) / ends (negative) per step.
fig, axes = plt.subplots(len(store_ids), 1, figsize=(11, 2.5 * len(store_ids)),
                          sharex=True, squeeze=False)
for i, sid in enumerate(store_ids):
    ax = axes[i, 0]
    g = decisions_per_store[decisions_per_store['store_id'] == sid]
    ax.bar(g['simulation_step'], g['promo_started'], color='#9467bd', label='Promo start')
    ax.bar(g['simulation_step'], -g['promo_ended'], color='#8c564b', label='Promo end')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_ylabel(f'Store {sid}\nevents', fontsize=9)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(loc='upper right', fontsize=9)
axes[-1, 0].set_xlabel('Simulation step')
fig.suptitle('Promotion starts / ends per step, per store')
fig.tight_layout()
plt.show()


In [ ]:
# Aggregate order quantity placed per step.
_per_store_panel(
    lambda ax, g: ax.bar(g['simulation_step'], g['order_qty'], color='#ff7f0e'),
    'Aggregate order quantity placed per step, per store', 'order qty',
)


In [ ]:
# Aggregate outstanding orders (units in transit).
_per_store_panel(
    lambda ax, g: ax.plot(g['simulation_step'], g['outstanding'], color='#1f77b4', lw=1.5),
    'Outstanding orders per step, per store', 'outstanding units',
)


In [ ]:
# Median realised price / base price across active SKUs. Reference at 1.0 = MSRP.
def _median_ratio(ax, g):
    ax.plot(g['simulation_step'], g['median_price_ratio'], color='black', lw=1.5,
            label='Median')
    ax.plot(g['simulation_step'], g['mean_price_ratio'], color='gray', lw=1, ls='--',
            label='Mean')
    ax.axhline(1.0, color='red', lw=0.6, ls=':')

fig, axes = plt.subplots(len(store_ids), 1, figsize=(11, 2.5 * len(store_ids)),
                          sharex=True, squeeze=False)
for i, sid in enumerate(store_ids):
    ax = axes[i, 0]
    g = decisions_per_store[decisions_per_store['store_id'] == sid]
    _median_ratio(ax, g)
    ax.set_ylabel(f'Store {sid}\nprice / MSRP', fontsize=9)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(loc='lower left', fontsize=9)
axes[-1, 0].set_xlabel('Simulation step')
fig.suptitle('Median active-SKU price / base price, per store')
fig.tight_layout()
plt.show()


## 5. Promotion + activation timelines (heatmaps)

One heatmap per store. Rows are SKUs ordered by total time spent in the
relevant state (promo / inactive). Rows with zero variance are dropped
so the figure reflects the SKUs the policy actually touched.

In [ ]:
promo_long = ts_sorted.assign(on_promo_int=ts_sorted['on_promo'].astype(int))
for sid in store_ids:
    g = promo_long[promo_long['store_id'] == sid]
    pivot = g.pivot_table(index='product_id', columns='simulation_step',
                          values='on_promo_int', aggfunc='max', fill_value=0)
    pivot['_total'] = pivot.sum(axis=1)
    pivot = pivot[pivot['_total'] > 0].sort_values('_total', ascending=False).drop(columns='_total')
    if pivot.empty:
        print(f'Store {sid}: no promotions ever fired.')
        continue
    fig, ax = plt.subplots(figsize=(11, max(3, 0.18 * len(pivot))))
    im = ax.imshow(pivot.values, aspect='auto', cmap='Purples',
                   interpolation='nearest',
                   extent=[pivot.columns.min(), pivot.columns.max(),
                           len(pivot), 0])
    ax.set_yticks(np.arange(len(pivot)) + 0.5)
    ax.set_yticklabels(pivot.index, fontsize=7)
    ax.set_xlabel('Simulation step')
    ax.set_title(f'Promotion timeline — store {sid} ({len(pivot)} SKUs ever promoted)')
    fig.tight_layout()
    plt.show()


In [ ]:
active_long = ts_sorted.assign(active_int=ts_sorted['active_status'].astype(int))
for sid in store_ids:
    g = active_long[active_long['store_id'] == sid]
    pivot = g.pivot_table(index='product_id', columns='simulation_step',
                          values='active_int', aggfunc='max', fill_value=0)
    n_steps = pivot.shape[1]
    # Drop rows that are constant — they add no signal.
    pivot = pivot[(pivot.sum(axis=1) > 0) & (pivot.sum(axis=1) < n_steps)]
    if pivot.empty:
        print(f'Store {sid}: every product stayed active for every step (no churn).')
        continue
    fig, ax = plt.subplots(figsize=(11, max(3, 0.18 * len(pivot))))
    ax.imshow(pivot.values, aspect='auto', cmap='Greens',
              interpolation='nearest',
              extent=[pivot.columns.min(), pivot.columns.max(),
                      len(pivot), 0])
    ax.set_yticks(np.arange(len(pivot)) + 0.5)
    ax.set_yticklabels(pivot.index, fontsize=7)
    ax.set_xlabel('Simulation step')
    ax.set_title(f'Activation timeline — store {sid} ({len(pivot)} SKUs with churn)')
    fig.tight_layout()
    plt.show()


## 6. Pricing dynamics

Per-store overlay of every SKU's realised price as a fraction of its MSRP,
with the median across active SKUs in bold. Reference line at 1.0 = MSRP.

In [ ]:
for sid in store_ids:
    g = ts[ts['store_id'] == sid].copy()
    g['price_ratio'] = g['price'] / g['base_price']
    ever_active = g.groupby('product_id')['active_status'].any()
    g = g[g['product_id'].isin(ever_active[ever_active].index)]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    # One line per product. Drop the legend (too many).
    for pid, sub in g.groupby('product_id'):
        ax.plot(sub['simulation_step'], sub['price_ratio'], alpha=0.55, lw=1.2,
                color='#1f77b4')
    med = (g[g['active_status'] == True]
           .groupby('simulation_step')['price_ratio'].median().reset_index())
    ax.plot(med['simulation_step'], med['price_ratio'], color='black', lw=2.5,
            label='Median (active SKUs)')
    ax.axhline(1.0, color='red', lw=1.0, ls='--', label='MSRP')
    ax.set_xlabel('Simulation step')
    ax.set_ylabel('Price / base price')
    ax.set_title(f'Price / base price per SKU — store {sid}')
    ax.legend(loc='lower left')
    ax.grid(True, alpha=0.4)
    fig.tight_layout()
    plt.show()

In [ ]:
# Discount distribution during promo windows, per store.
promo_rows = ts[ts['promotion_status'] != 'Regular Price'].copy()
promo_rows['discount_pct'] = (1 - promo_rows['price'] / promo_rows['base_price']) * 100
if promo_rows.empty:
    print('No promotion rows — skipping discount distribution.')
else:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    data = [promo_rows[promo_rows['store_id'] == sid]['discount_pct'].values
            for sid in store_ids]
    bp = ax.boxplot(data, labels=[f'Store {sid}' for sid in store_ids],
                    patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.tab10(np.linspace(0, 1, len(store_ids)))):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel('1 - price/base_price (%)')
    ax.set_title('Realised discount % during promotions — per store')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


## 7. Inventory / sales / demand flow per store

Aggregate per-step flow rates split into two figures so each has room to
breathe: stock + replenishment pipeline, then demand vs sales (unmet
demand shaded).

In [ ]:
flow = (ts.groupby(['store_id', 'simulation_step'], as_index=False)
          .agg(inventory=('inventory', 'sum'),
               demand=('demand', 'sum'),
               sales=('sales', 'sum'),
               order_qty=('order_quantity', 'sum'),
               outstanding=('outstanding_orders', 'sum')))

for sid in store_ids:
    g = flow[flow['store_id'] == sid]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(g['simulation_step'], g['inventory'], color='#1f77b4', lw=2.0,
            label='Inventory')
    ax.plot(g['simulation_step'], g['outstanding'], color='#ff7f0e', lw=2.0,
            label='Outstanding')
    ax.bar(g['simulation_step'], g['order_qty'], color='#2ca02c', alpha=0.85,
           label='Order qty')
    ax.set_xlabel('Simulation step')
    ax.set_ylabel('Units')
    ax.set_title(f'Store {sid} — inventory, outstanding, orders')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.4)
    fig.tight_layout()
    plt.show()

for sid in store_ids:
    g = flow[flow['store_id'] == sid]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(g['simulation_step'], g['demand'], color='#9467bd', lw=2.0, ls='--',
            label='Demand')
    ax.plot(g['simulation_step'], g['sales'], color='#2ca02c', lw=2.0,
            label='Sales')
    ax.fill_between(g['simulation_step'], g['sales'], g['demand'],
                    where=(g['demand'] > g['sales']),
                    color='#d62728', alpha=0.35, label='Unmet demand')
    ax.set_xlabel('Simulation step')
    ax.set_ylabel('Units')
    ax.set_title(f'Store {sid} — demand vs sales (unmet shaded)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.4)
    fig.tight_layout()
    plt.show()

## 8. Per-(store, product) drill-down

`drilldown(store, product_id)` renders a multi-panel figure for one SKU:
inventory + outstanding + orders, demand vs sales, price / base / unit
cost, cumulative profit, and active / on-promo state strips.

The default cell below picks the best- and worst-profit SKUs in the first
store. Call `drilldown(store, pid)` for any (store, product) you want.

In [ ]:
def drilldown(store_id, product_id):
    g = ts[(ts['store_id'] == store_id) & (ts['product_id'] == product_id)].copy()
    g = g.sort_values('simulation_step')
    if g.empty:
        print(f'No rows for store={store_id} product={product_id}')
        return
    g['cum_profit'] = g['profit'].cumsum()
    name = g['name'].iloc[0]
    cat = g['category'].iloc[0]
    fig, axes = plt.subplots(5, 1, figsize=(11, 11), sharex=True,
                              gridspec_kw={'height_ratios': [3, 3, 3, 2, 1]})
    ax0, ax1, ax2, ax3, ax4 = axes
    # Inventory / outstanding / orders.
    ax0.plot(g['simulation_step'], g['inventory'], color='#1f77b4', lw=2.0,
             label='Inventory')
    ax0.plot(g['simulation_step'], g['outstanding_orders'], color='#ff7f0e', lw=2.0,
             label='Outstanding')
    ax0.bar(g['simulation_step'], g['order_quantity'], color='#2ca02c', alpha=0.85,
            label='Order qty')
    ax0.set_ylabel('Units')
    ax0.legend(loc='upper right', fontsize=8)
    ax0.grid(True, alpha=0.4)
    # Demand vs sales.
    ax1.plot(g['simulation_step'], g['demand'], color='#9467bd', lw=2.0, ls='--', label='Demand')
    ax1.plot(g['simulation_step'], g['sales'], color='#2ca02c', lw=2.0, label='Sales')
    ax1.fill_between(g['simulation_step'], g['sales'], g['demand'],
                     where=(g['demand'] > g['sales']),
                     color='#d62728', alpha=0.35, label='Unmet')
    ax1.set_ylabel('Units')
    ax1.legend(loc='upper right', fontsize=8)
    ax1.grid(True, alpha=0.4)
    # Price / base / cost.
    ax2.plot(g['simulation_step'], g['price'], color='black', lw=2.0, label='Price')
    ax2.plot(g['simulation_step'], g['base_price'], color='gray', lw=1.5, ls='--', label='Base price')
    ax2.plot(g['simulation_step'], g['unit_cost'], color='red', lw=1.5, ls=':', label='Unit cost')
    ax2.set_ylabel('Price')
    ax2.legend(loc='upper right', fontsize=8)
    ax2.grid(True, alpha=0.4)
    # Cumulative profit.
    ax3.plot(g['simulation_step'], g['cum_profit'], color='crimson', lw=2.2)
    ax3.axhline(0, color='gray', lw=0.5)
    ax3.set_ylabel('Cum. profit')
    ax3.grid(True, alpha=0.4)
    # State strip: active (green) / promo (purple).
    ax4.fill_between(g['simulation_step'], 0, g['active_status'].astype(int),
                     step='post', color='#2ca02c', alpha=0.6, label='Active')
    ax4.fill_between(g['simulation_step'], 0,
                     (g['promotion_status'] != 'Regular Price').astype(int),
                     step='post', color='#9467bd', alpha=0.85, label='On promo')
    ax4.set_ylim(0, 1.1)
    ax4.set_yticks([0, 1])
    ax4.legend(loc='upper right', fontsize=8)
    ax4.set_xlabel('Simulation step')
    fig.suptitle(f'Drill-down — store {store_id} / {product_id} ({name}, {cat})')
    fig.tight_layout()
    plt.show()


profits_s0 = (ts[ts['store_id'] == store_ids[0]]
              .groupby('product_id')['profit'].sum().sort_values())
best = profits_s0.idxmax()
worst = profits_s0.idxmin()
print(f'Best SKU in store {store_ids[0]}: {best} (profit={profits_s0[best]:.0f})')
print(f'Worst SKU in store {store_ids[0]}: {worst} (profit={profits_s0[worst]:.0f})')
drilldown(store_ids[0], best)
drilldown(store_ids[0], worst)

## 9. Product- and category-level profitability per store

For each store: bar chart of the top 15 most-profitable SKUs and the
bottom 15 (worst-loss) SKUs. Then a category × store P&L heatmap.

In [ ]:
prod_perf = (ts.groupby(['store_id', 'product_id'], as_index=False)
               .agg(revenue=('revenue', 'sum'),
                    profit=('profit', 'sum'),
                    sales=('sales', 'sum'))
               .merge(products_df[['product_id', 'name', 'category']], on='product_id'))

# Stable color per category across all per-store figures.
categories = sorted(prod_perf['category'].unique())
cat_colors = dict(zip(categories, cm.tab20(np.linspace(0, 1, len(categories)))))

for sid in store_ids:
    g = prod_perf[prod_perf['store_id'] == sid].sort_values('profit')
    bot = g.head(15)
    top = g.tail(15)
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.5))
    axL.barh(bot['name'], bot['profit'],
             color=[cat_colors[c] for c in bot['category']])
    axL.set_title(f'Store {sid} — bottom 15 (worst P&L)')
    axL.axvline(0, color='gray', lw=0.5)
    axL.tick_params(axis='y', labelsize=8)
    axR.barh(top['name'], top['profit'],
             color=[cat_colors[c] for c in top['category']])
    axR.set_title(f'Store {sid} — top 15 (best P&L)')
    axR.axvline(0, color='gray', lw=0.5)
    axR.tick_params(axis='y', labelsize=8)
    handles = [Patch(color=cat_colors[c], label=c) for c in categories]
    fig.legend(handles=handles, loc='center right', fontsize=7,
               bbox_to_anchor=(1.0, 0.5))
    fig.tight_layout(rect=[0, 0, 0.88, 1])
    plt.show()


In [ ]:
cat_perf = (prod_perf.groupby(['store_id', 'category'], as_index=False)['profit'].sum()
            .pivot(index='category', columns='store_id', values='profit').fillna(0))
fig, ax = plt.subplots(figsize=(6, max(3, 0.35 * len(cat_perf))))
vmax = float(np.abs(cat_perf.values).max() or 1.0)
im = ax.imshow(cat_perf.values, aspect='auto', cmap='RdYlGn',
               vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(cat_perf.columns)))
ax.set_xticklabels([f'Store {s}' for s in cat_perf.columns])
ax.set_yticks(range(len(cat_perf.index)))
ax.set_yticklabels(cat_perf.index, fontsize=9)
for i in range(cat_perf.shape[0]):
    for j in range(cat_perf.shape[1]):
        ax.text(j, i, f'{cat_perf.values[i, j]:.0f}',
                ha='center', va='center',
                color='black', fontsize=8)
ax.set_title('Category P&L per store')
fig.colorbar(im, ax=ax, label='Cumulative profit')
fig.tight_layout()
plt.show()


## 10. Fulfilment quality — demand vs sales

Per-(store, product) fill rate (sales / demand). Scatter shows whether
sales tracked demand (`fill_rate ≈ 1`) or got capped by stockouts
(`fill_rate < 1`). Marker size is unmet-demand magnitude.

In [ ]:
mask = ts['demand'] > 0
fill = (ts[mask].groupby(['store_id', 'product_id'], as_index=False)
         .agg(total_demand=('demand', 'sum'),
              total_sales=('sales', 'sum'))
         .merge(products_df[['product_id', 'category']], on='product_id'))
fill['fill_rate'] = fill['total_sales'] / fill['total_demand']
fill['unmet'] = fill['total_demand'] - fill['total_sales']

for sid in store_ids:
    g = fill[fill['store_id'] == sid]
    fig, ax = plt.subplots(figsize=(9, 5))
    sizes = (g['unmet'].clip(lower=1).pow(0.5) * 3).clip(lower=10, upper=400)
    sc = ax.scatter(g['total_demand'], g['fill_rate'],
                    s=sizes, alpha=0.55,
                    c=[cat_colors[c] for c in g['category']])
    ax.axhline(1.0, color='gray', lw=0.7, ls='--')
    ax.set_xlabel('Total demand (whole run)')
    ax.set_ylabel('Fill rate = sales / demand')
    ax.set_title(f'Store {sid} — per-SKU fill rate vs demand')
    ax.grid(True, alpha=0.3)
    handles = [Patch(color=cat_colors[c], label=c) for c in categories
               if c in g['category'].unique()]
    ax.legend(handles=handles, fontsize=7, loc='lower right', ncol=2)
    fig.tight_layout()
    plt.show()


In [ ]:
# Fill-rate distribution by category, per store.
for sid in store_ids:
    g = fill[fill['store_id'] == sid]
    if g.empty:
        continue
    fig, ax = plt.subplots(figsize=(11, 4.5))
    cats_present = [c for c in categories if c in g['category'].unique()]
    data = [g[g['category'] == c]['fill_rate'].values for c in cats_present]
    bp = ax.boxplot(data, labels=cats_present, patch_artist=True,
                    flierprops=dict(marker='.', markersize=3))
    for patch, c in zip(bp['boxes'], cats_present):
        patch.set_facecolor(cat_colors[c])
        patch.set_alpha(0.6)
    ax.axhline(1.0, color='gray', lw=0.7, ls='--')
    ax.set_ylabel('Fill rate')
    ax.set_title(f'Store {sid} — fill-rate distribution by category')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


## 11. Final snapshot

In [ ]:
final_step = ts['simulation_step'].max()
final = ts[ts['simulation_step'] == final_step]
final_per_store = (final.groupby('store_id')
                   .agg(inventory_units=('inventory', 'sum'),
                        outstanding_units=('outstanding_orders', 'sum'),
                        active_skus=('active_status', 'sum'),
                        on_promo_skus=('promotion_status', lambda s: (s != 'Regular Price').sum()))
                   .reset_index())
final_per_store = final_per_store.merge(
    store_step[store_step['simulation_step'] == final_step][[
        'store_id', 'cash', 'inventory_value', 'outstanding_value',
        'equity', 'cumulative_profit']],
    on='store_id')
final_per_store


In [ ]:
totals = pd.Series({
    'simulation_folder': str(latest_sim_dir),
    'n_stores': int(ts['store_id'].nunique()),
    'n_products': int(ts['product_id'].nunique()),
    'n_steps': int(ts['simulation_step'].nunique()),
    'date_range': f'{sim_dates.min().date()} → {sim_dates.max().date()}',
    'total_revenue': float(ts['revenue'].sum()),
    'total_cost': float(ts['total_cost'].sum()),
    'total_profit': float(ts['profit'].sum()),
    'total_promo_steps': int((ts['promotion_status'] != 'Regular Price').sum()),
    'total_activations': int(ts_sorted['activated'].sum()),
    'total_deactivations': int(ts_sorted['deactivated'].sum()),
}, name='run_summary')
totals
